# 01 · Target Synthesis & Train/Test Split
**Binary label:** `target_risk = 1` when `ndvi_mean < county 25th percentile` (computed on train 2015-2023 only). Chronological split: 2015-2023 train (5076) / 2024-2025 test (1128).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../../data_pipeline/data/processed")

# Load ml_base with fallback
src = DATA_DIR / "ml_base_2015_2025.csv"
if not src.exists():
    src = DATA_DIR / "master_cleaned_2015_2025.csv"
df = pd.read_csv(src)
df = df.rename(columns={"temperature_mean":"temp_mean_c","rainfall_sum":"rainfall_mm","soil_moisture_mean":"soil_moisture"})
if "year_month" not in df.columns:
    df["year_month"] = df["year"].astype(str) + "-" + df["month"].astype(str).str.zfill(2)

df = df.sort_values(["county","year","month"]).reset_index(drop=True)
print(f"Base: {len(df)} rows | {df['county'].nunique()} counties | {df['year'].min()}-{df['year'].max()}")
display(df.head())


In [ ]:
# Remove old target if re-running
df = df.drop(columns=[c for c in ["ndvi_25th_pct","ndvi_mean_hist","ndvi_std_hist","target_risk","rainfall_anomaly","rain_hist"] if c in df.columns])

# Split before any threshold (no leakage)
train = df[df["year"] <= 2023].copy()
test = df[df["year"] >= 2024].copy()
print(f"Train 2015-2023: {len(train)} | Test 2024-2025: {len(test)}")

# County thresholds on TRAIN only (25th percentile)
stats = train.groupby("county", as_index=False).agg(
    ndvi_25th_pct=("ndvi_mean", lambda s: float(np.percentile(s, 25))),
    ndvi_mean_hist=("ndvi_mean", "mean"),
    ndvi_std_hist=("ndvi_mean", "std"),
)
df = df.merge(stats, on="county", how="left")
df["target_risk"] = (df["ndvi_mean"] < df["ndvi_25th_pct"]).astype(int)

# Rainfall anomaly: deviation from train county-month mean
rain_hist = train.groupby(["county","month"], as_index=False)["rainfall_mm"].mean().rename(columns={"rainfall_mm":"rain_hist"})
df = df.merge(rain_hist, on=["county","month"], how="left")
df["rainfall_anomaly"] = (df["rainfall_mm"] - df["rain_hist"]).round(4)
df = df.drop(columns=["rain_hist"])

print(f"Train shock: {df[df['year']<=2023]['target_risk'].mean():.1%} | Test shock: {df[df['year']>=2024]['target_risk'].mean():.1%}")
display(stats.head())


In [ ]:
# Save
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(DATA_DIR / "ml_base_2015_2025.csv", index=False)
df[df["year"]<=2023].to_csv(DATA_DIR / "master_train_2015_2023.csv", index=False)
df[df["year"]>=2024].to_csv(DATA_DIR / "master_test_2024_2025.csv", index=False)
stats.to_csv(DATA_DIR / "county_ndvi_stats.csv", index=False)
stats.to_csv(MODEL_DIR / "county_ndvi_stats.csv", index=False)
print(f"✓ Saved ml_base ({len(df)}), train ({len(df[df['year']<=2023])}), test ({len(df[df['year']>=2024])}) + county stats")
